# Sentence Transformers to create vector embeddings
#### Note: Requires Python >= 3.10

## Import packages.

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## Read in csv file as a Pandas DataFrame and define the Sentence Transformer model.

In [2]:
df = pd.read_csv("../data/raw/movies.csv")
model = SentenceTransformer("all-MiniLM-L6-v2") # Source: https://sbert.net/docs/sentence_transformer/usage/usage.html

/opt/conda/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Create a text column to combine "title", "genres", "tagline", and "overview".

In [81]:
df_text = df[['title', 'genres', 'tagline', 'overview']]
df_text['summarized_text'] = ""
df_text

/tmp/ipykernel_97/710830393.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_text['summarized_text'] = ""


,title,genres,tagline,overview,summarized_text
0,Avatar,Action Adventure Fantasy Science Fiction,Enter the World of Pandora.,"In the 22nd century, a paraplegic Marine is di...",
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,"At the end of the world, the adventure begins.","Captain Barbossa, long believed to be dead, ha...",
2,Spectre,Action Adventure Crime,A Plan No One Escapes,A cryptic message from Bond’s past sends him o...,
3,The Dark Knight Rises,Action Crime Drama Thriller,The Legend Ends,Following the death of District Attorney Harve...,
4,John Carter,Action Adventure Science Fiction,"Lost in our world, found in another.","John Carter is a war-weary, former military ca...",
...,...,...,...,...,...
4798,El Mariachi,Action Crime Thriller,"He didn't come looking for trouble, but troubl...",El Mariachi just wants to play his guitar and ...,
4799,Newlyweds,Comedy Romance,A newlywed couple's honeymoon is upended by th...,A newlywed couple's honeymoon is upended by th...,
4800,"Signed, Sealed, Delivered",Comedy Drama Romance TV Movie,NaN,"""Signed, Sealed, Delivered"" introduces a dedic...",
4801,Shanghai Calling,NaN,A New Yorker in Shanghai,When ambitious New York attorney Sam is sent t...,


In [82]:
def words_to_list(string):
    text_to_return = ""
    for count, word in enumerate(string.split()):
        text_to_return += word.lower()
        if count < len(string.split()) - 1:
            text_to_return += ", "
    return text_to_return

In [83]:
words_to_list(df['genres'][0])

'action, adventure, fantasy, science, fiction'

In [84]:
def format_text(title, genres, tagline, overview):
    if pd.isna(title):
        title = "Untitled"
    if pd.isna(genres):
        genres = "None"
    if pd.isna(tagline):
        tagline = ""
    if pd.isna(overview):
        overview = ""
    return f"{title} (Genres: {words_to_list(genres)}): {tagline} {overview}"

In [85]:
format_text(df_text.iloc[0][0], df_text.iloc[0][1], df_text.iloc[0][2], df_text.iloc[0][3])

'Avatar (Genres: action, adventure, fantasy, science, fiction): Enter the World of Pandora. In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [86]:
format_text(df_text.iloc[10][0], df_text.iloc[10][1], df_text.iloc[10][2], df_text.iloc[10][3])

'Superman Returns (Genres: adventure, fantasy, action, science, fiction):  Superman returns to discover his 5-year absence has allowed Lex Luthor to walk free, and that those he was closest too felt abandoned and have moved on. Luthor plots his ultimate revenge that could see millions killed and change the face of the planet forever, as well as ridding himself of the Man of Steel.'

In [87]:
summarized_text = []
for i in range(df_text.shape[0]):
    row = df.iloc[i]
    summarized_text.append(format_text(df_text.iloc[i][0], 
                                            df_text.iloc[i][1], 
                                            df_text.iloc[i][2], 
                                            df_text.iloc[i][3]))
df_text['summarized_text'] = summarized_text
df_text

/tmp/ipykernel_97/2216420655.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_text['summarized_text'] = summarized_text


,title,genres,tagline,overview,summarized_text
0,Avatar,Action Adventure Fantasy Science Fiction,Enter the World of Pandora.,"In the 22nd century, a paraplegic Marine is di...","Avatar (Genres: action, adventure, fantasy, sc..."
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,"At the end of the world, the adventure begins.","Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End (Genr...
2,Spectre,Action Adventure Crime,A Plan No One Escapes,A cryptic message from Bond’s past sends him o...,"Spectre (Genres: action, adventure, crime): A ..."
3,The Dark Knight Rises,Action Crime Drama Thriller,The Legend Ends,Following the death of District Attorney Harve...,"The Dark Knight Rises (Genres: action, crime, ..."
4,John Carter,Action Adventure Science Fiction,"Lost in our world, found in another.","John Carter is a war-weary, former military ca...","John Carter (Genres: action, adventure, scienc..."
...,...,...,...,...,...
4798,El Mariachi,Action Crime Thriller,"He didn't come looking for trouble, but troubl...",El Mariachi just wants to play his guitar and ...,"El Mariachi (Genres: action, crime, thriller):..."
4799,Newlyweds,Comedy Romance,A newlywed couple's honeymoon is upended by th...,A newlywed couple's honeymoon is upended by th...,"Newlyweds (Genres: comedy, romance): A newlywe..."
4800,"Signed, Sealed, Delivered",Comedy Drama Romance TV Movie,NaN,"""Signed, Sealed, Delivered"" introduces a dedic...","Signed, Sealed, Delivered (Genres: comedy, dra..."
4801,Shanghai Calling,NaN,A New Yorker in Shanghai,When ambitious New York attorney Sam is sent t...,Shanghai Calling (Genres: none): A New Yorker ...


## Get vector embeddings of the summarized text column.

In [91]:
text = df_text["summarized_text"]

In [92]:
embeddings = model.encode(text) # Result in a 1D array with 384 elements.

In [93]:
# 4803 rows and 384 columns.
embeddings_array = np.array(embeddings)
print(embeddings_array.shape)

(4803, 384)


## Convert to a DataFrame and csv file.

In [94]:
vector_df = pd.DataFrame(embeddings_array)
vector_df

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,0.041741,0.009301,0.051212,-0.038178,-0.017390,0.007238,0.033966,-0.059363,0.050549,0.062952,...,0.071803,-0.016974,-0.023629,0.055914,0.027464,0.029297,0.020496,0.022388,-0.000118,-0.015742
1,-0.024948,-0.002718,-0.047993,0.009058,-0.006901,0.077876,-0.009504,-0.013634,0.066740,0.052810,...,0.070627,0.010297,-0.051285,0.168123,0.000585,0.026403,0.038032,-0.097817,-0.047968,0.000822
2,-0.038338,0.014658,-0.045681,0.002359,0.019390,0.012511,0.042053,0.046065,0.026654,0.042536,...,0.000143,0.056230,-0.040874,0.005279,-0.061348,-0.036539,0.060177,-0.037475,-0.037140,-0.007060
3,-0.013254,-0.025280,-0.073584,0.008209,0.029183,0.057144,-0.055012,0.039153,0.061290,0.017189,...,-0.035736,-0.010435,-0.015904,0.029208,-0.036897,-0.013742,0.091292,-0.054864,0.052357,0.056075
4,-0.007292,-0.012704,-0.005583,-0.002228,-0.016023,0.050177,0.081254,0.003447,0.036348,0.085831,...,0.163694,-0.007840,-0.033616,0.136025,0.014909,-0.049623,-0.041421,-0.047612,-0.090229,-0.059567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,-0.016934,0.026891,-0.009934,-0.013264,0.019712,0.046333,0.039210,0.077309,0.022972,-0.032818,...,0.046814,-0.020037,0.013356,0.059392,-0.016953,-0.039655,-0.022799,0.009537,0.026096,-0.011732
4799,-0.012558,-0.071720,-0.000545,0.007462,-0.066228,0.062044,-0.019284,-0.074545,0.058040,0.049183,...,0.016861,0.023705,-0.014548,0.104421,0.050119,0.067815,-0.006594,-0.017928,0.063170,-0.022428
4800,-0.082294,-0.030497,-0.017078,-0.019795,-0.067209,0.011513,0.036340,-0.043270,0.018085,0.009315,...,0.060841,0.015467,0.062204,0.073132,0.030850,0.025292,-0.033975,0.001327,0.038575,-0.026509
4801,-0.050002,-0.074313,-0.022521,-0.012435,-0.102164,0.011200,0.099728,-0.069037,0.058911,-0.091959,...,0.001358,-0.015414,0.011845,0.008552,0.003379,0.071455,0.079651,-0.028741,-0.049205,0.084100


In [95]:
vector_df.to_csv("../data/interim/Summarized_Text_2.0_Vector_Embeddings.csv")